In [1]:
import pandas as pd
import numpy as np


dataset = pd.read_csv('data/flats_final.csv')

In [2]:
dataset.sample(frac = 1)

,floor,floors_count,rooms_count,total_meters,price,distance_to_subway,distance_to_center,district_rank
3760,3,5,4,49.00,8000000,0.004829,10354.637297,17
1399,7,23,2,52.43,10637648,0.000000,13695.060399,9
2528,6,9,3,114.30,30861000,0.000882,8105.376344,6
2502,2,6,3,87.73,29439170,0.000000,29304.356602,5
3779,1,9,4,76.00,9000000,0.000612,10370.344532,15
...,...,...,...,...,...,...,...,...
2902,4,9,4,73.30,11300000,0.000715,10366.218878,6
4903,11,19,-1,22.90,5000000,0.000227,6283.947924,4
1956,10,12,3,121.00,39963000,0.001196,5006.838570,9
1247,13,22,2,50.70,8137350,0.000215,12489.109860,10


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# Создаем кадр данных pandas с тремя столбцами 'A', 'B', 'C'
# np.random.seed(10)
data = dataset

# Определяем выбросы с использованием метода Z-оценки
z = np.abs(stats.zscore(data))
data_clean_z = data[(z < 3).all(axis=1)]

# Определяем выбросы с использованием межквартильного размаха
Q1 = data.quantile(q=0.25)
Q3 = data.quantile(q=0.75)
IQR = data.apply(stats.iqr)
data_clean_iqr = data[~((data < (Q1 - 1.5 * IQR)) | (data > (Q3 + 1.5 * IQR))).any(axis=1)]

# Выводим количество оставшихся строк в кадре данных
print("Количество строк после удаления выбросов (метод Z-оценки):", data_clean_z.shape[0])
print("Количество строк после удаления выбросов (метод межквартильного размаха):", data_clean_iqr.shape[0])



Количество строк после удаления выбросов (метод Z-оценки): 4622
Количество строк после удаления выбросов (метод межквартильного размаха): 4029


In [5]:
y = data_clean_iqr['price']
X = data_clean_iqr.drop(['price'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)



In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

In [8]:
X_train.shape

(4016, 7)

In [7]:
from torch import nn

class model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(7, 32),   
            nn.BatchNorm1d(32),
            nn.ReLU()
        )
        self.layer2 = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )
        self.layer1 = nn.Sequential(
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.final = nn.Linear(128, 1)
    def forward(self, x):
        return self.final(self.layer3(self.layer2(self.layer1(x))))




array([[-0.35753157, -0.99895361,  1.23019588, ...,  1.35326283,
        -0.97626072, -1.23350566],
       [ 0.25329378,  0.64528703, -0.87571489, ..., -0.56161389,
         1.0447283 ,  1.30598183],
       [-0.56114002, -0.99895361,  1.23019588, ..., -0.53838734,
         0.47191655,  1.69667222],
       ...,
       [-0.76474847, -0.40104792,  0.52822563, ..., -0.50249735,
        -0.82589712, -1.03816046],
       [-0.56114002, -0.99895361, -0.87571489, ...,  1.92619377,
        -0.95681488, -1.23350566],
       [-0.15392312, -0.99895361, -0.87571489, ..., -0.71025845,
         1.45910858, -0.2567797 ]])

In [5]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

LinearRegression()

In [6]:
y_pred = lin_reg.predict(X_test)

deviation = np.abs(y_test - y_pred)

# проверяем, сколько предсказаний отклоняется более чем на 500000
deviation_above_threshold = deviation[deviation > 500000]

# выводим количество предсказаний, которые отклоняются более чем на 500000
print(f"Number of predictions that deviate more than 500000: {len(deviation_above_threshold)}")
print(len(y_test) - len(deviation_above_threshold))

Number of predictions that deviate more than 500000: 947
57


In [7]:
dec_tree = DecisionTreeRegressor(max_depth=20)

dec_tree.fit(X_train, y_train)


DecisionTreeRegressor(max_depth=20)

In [8]:
y_pred = dec_tree.predict(X_test)

deviation = np.abs(y_test - y_pred)

# проверяем, сколько предсказаний отклоняется более чем на 500000
deviation_above_threshold = deviation[deviation > 500000]

# выводим количество предсказаний, которые отклоняются более чем на 500000
print(f"Number of predictions that deviate more than 500000: {len(deviation_above_threshold)}")
print(len(y_test) - len(deviation_above_threshold))

Number of predictions that deviate more than 500000: 686
318


In [4]:
rand_for = RandomForestRegressor(max_depth=30)
rand_for.fit(X_train, y_train)

RandomForestRegressor(max_depth=30)

In [5]:
y_pred = rand_for.predict(X_test)

deviation = np.abs(y_test - y_pred)

# проверяем, сколько предсказаний отклоняется более чем на 500000
deviation_above_threshold = deviation[deviation > 500000]

# выводим количество предсказаний, которые отклоняются более чем на 500000
print(f"Number of predictions that deviate more than 500000: {len(deviation_above_threshold)}")
print(len(y_test) - len(deviation_above_threshold))

Number of predictions that deviate more than 500000: 702
302


У линейной регрессии есть отрицательные предсказания, у деревьев и рандомного леса нет!!!

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)


In [7]:
X_test_scaled = scaler.transform(X_test)

In [39]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd


# конвертируем данные в тензоры PyTorch
X_train_t = torch.FloatTensor(X_train_scaled)
X_test_t = torch.FloatTensor(X_test_scaled)
y_train_t = torch.FloatTensor(y_train.to_numpy())
y_test_t = torch.FloatTensor(y_test.to_numpy())

# создаем DataLoader'ы для обучения и тестирования
train_data = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_data, batch_size=8)
test_data = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_data, batch_size=1)

In [38]:
# определяем модель

class Net(nn.Module):
    def __init__(self, input_size):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(7, 30)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(30, 120)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(120, 10)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(10, 1)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.relu3(out)
        out = self.fc4(out)
        return out

In [40]:
model = Net(7)

# определяем функцию потерь и оптимизатор
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

# обучаем модель
for epoch in range(10):
    total_loss = 0
    for inputs, targets in train_loader:
        # обнуляем градиенты
        optimizer.zero_grad()
        # прямой проход
        outputs = model(inputs)
        # вычисляем потери
        loss = criterion(outputs, targets)
        loss = torch.sqrt(loss)
        total_loss += loss.item()
        # обратный проход
        loss.backward()
        # обновляем веса
        optimizer.step()
    if epoch % 1 == 0:
        print(f'Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}')

# оцениваем модель
model.eval()
with torch.no_grad():
    total_loss = 0
    for inputs, targets in test_loader:
        outputs = model(inputs)
        total_loss += 1 if abs(outputs - targets) <= 500000 else 0
    print(f'Average loss: {total_loss}')

Epoch 1, Loss: 10382705.503722085
Epoch 2, Loss: 8749403.277295286
Epoch 3, Loss: 8738671.183002481
Epoch 4, Loss: 8736172.441066997
Epoch 5, Loss: 8734865.282258065
Epoch 6, Loss: 8734229.772332506
Epoch 7, Loss: 8733849.830024814
Epoch 8, Loss: 8733543.879032258
Epoch 9, Loss: 8733241.376550868
Epoch 10, Loss: 8732946.571960298
Average loss: 33


170 с фциями активации


In [40]:
outputs, targets

(tensor([[6037843.5000]]), tensor([5800000.]))

In [54]:
model.eval()
with torch.no_grad():
    total_loss = 0
    for inputs, targets in test_loader:
        outputs = model(inputs)
        total_loss += 1 if abs(outputs - targets) <= 10000000 else 0
    print(f'Average loss: {total_loss}')

Average loss: 882


In [45]:
len(test_loader)

1004

In [7]:
from sklearn.svm import SVR

In [9]:
svr = SVR()

svr.fit(X_train, y_train)

SVR()

In [10]:
y_pred = svr.predict(X_test)

from sklearn.metrics import mean_absolute_percentage_error
 
print(mean_absolute_percentage_error(y_test, y_pred))

0.5621502460416224


In [11]:
y_pred = rand_for.predict(X_test)

print(mean_absolute_percentage_error(y_test, y_pred))

0.18044418575822838


In [12]:
deviation = np.abs(y_test - y_pred)

# проверяем, сколько предсказаний отклоняется более чем на 500000
deviation_above_threshold = deviation[deviation > 500000]

# выводим количество предсказаний, которые отклоняются более чем на 500000
print(f"Number of predictions that deviate more than 500000: {len(deviation_above_threshold)}")
print(len(y_test) - len(deviation_above_threshold))

Number of predictions that deviate more than 500000: 702
302


In [ ]:
from catboost import CatBoostRegressor
 
# Initialize the CatBoostRegressor with RMSE as the loss function
model = CatBoostRegressor(loss_function='RMSE', learning_rate=0.1, iterations=3000)
 
# Fit the model on the training data with verbose logging every 100 iterations
model.fit(X, y, verbose=100)

0:	learn: 26878453.9667004	total: 37.2ms	remaining: 1m 51s
100:	learn: 9086926.4828998	total: 612ms	remaining: 17.6s
200:	learn: 7070003.7956846	total: 1.07s	remaining: 14.9s
300:	learn: 5797501.2940319	total: 1.55s	remaining: 13.9s
400:	learn: 5035162.3225438	total: 1.95s	remaining: 12.6s
500:	learn: 4504163.0513878	total: 2.39s	remaining: 11.9s
600:	learn: 4064190.1770020	total: 2.82s	remaining: 11.2s
700:	learn: 3742763.7521957	total: 3.26s	remaining: 10.7s
800:	learn: 3415920.4807112	total: 3.74s	remaining: 10.3s
900:	learn: 3161920.8419153	total: 4.17s	remaining: 9.71s
1000:	learn: 2936744.2356652	total: 4.58s	remaining: 9.15s
1100:	learn: 2745299.6422310	total: 5.24s	remaining: 9.04s


In [50]:
y_pred = model.predict(X_test)

print(mean_absolute_percentage_error(y_test, y_pred))

0.15963292766418788


In [51]:
deviation = np.abs(y_test - y_pred)

# проверяем, сколько предсказаний отклоняется более чем на 500000
deviation_above_threshold = deviation[deviation > 500000]

# выводим количество предсказаний, которые отклоняются более чем на 500000
print(f"Number of predictions that deviate more than 500000: {len(deviation_above_threshold)}")
print(len(y_test) - len(deviation_above_threshold))

Number of predictions that deviate more than 500000: 682
322
